# OE4 — Análisis Pretest / Postest (t de Student pareada)

Tesis: Tutor con IA generativa — Aplicaciones Móviles · IESTP RFA.

> ⚠️ **ADVERTENCIA:** este notebook valida el *pipeline* de análisis. Mientras no se carguen los **datos reales** del piloto (Sprint 8), usa una **simulación claramente rotulada** que **NO** constituye evidencia del OE4. Reemplazar por los datos reales antes de la sustentación.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Cargar datos reales del piloto (si existen)

Completar `docs/datos-pretest-postest.csv` con columnas `codigo_estudiante,pretest,postest` (puntajes 0–20).

In [ ]:
CSV = Path('../../docs/datos-pretest-postest.csv')
if CSV.exists():
    df = pd.read_csv(CSV).dropna()
    df = df if len(df) > 0 else None
else:
    df = None

if df is not None:
    print(f'Datos reales del piloto: {len(df)} estudiantes')
else:
    print('Aún no hay datos reales. Se usará la simulación del pipeline (celda 2).')

## 2. Validación del pipeline con DATOS SIMULADOS

**NO son resultados reales ni evidencia del OE4.** Solo comprueban que el análisis corre.

In [ ]:
# ======================================================================
# DATOS SIMULADOS — sintéticos, solo para validar el pipeline.
# Reemplazar por los datos reales del piloto (celda 1) en Sprint 8.
# ======================================================================
if df is None:
    rng = np.random.default_rng(42)
    n = 15
    pre_sim = np.clip(rng.normal(11, 3, n), 0, 20).round(0)
    post_sim = np.clip(pre_sim + rng.normal(3.5, 2, n), 0, 20).round(0)
    df = pd.DataFrame({
        'codigo_estudiante': [f'SIM{i+1:02d}' for i in range(n)],
        'pretest': pre_sim,
        'postest': post_sim,
    })
    print('*** USANDO DATOS SIMULADOS (no reales) ***')
df

## 3. Estadística descriptiva

In [ ]:
pre = df['pretest'].astype(float)
post = df['postest'].astype(float)
diff = post - pre
print(df[['pretest', 'postest']].describe().round(2))
print(f'\nMedia pretest  = {pre.mean():.2f}')
print(f'Media postest  = {post.mean():.2f}')
print(f'Diferencia media = {diff.mean():.2f} (DE = {diff.std(ddof=1):.2f})')

## 4. Normalidad de las diferencias (Shapiro-Wilk)

In [ ]:
w, p_norm = stats.shapiro(diff)
print(f'Shapiro-Wilk: W = {w:.3f}, p = {p_norm:.4f}')
print('Normalidad OK -> t pareada' if p_norm >= 0.05 else 'No normal -> usar Wilcoxon')

## 5. Contraste: t de Student pareada + tamaño del efecto

In [ ]:
t, p = stats.ttest_rel(post, pre, alternative='greater')
d = diff.mean() / diff.std(ddof=1)  # d de Cohen para muestras relacionadas
gl = len(df) - 1
print(f't de Student pareada: t({gl}) = {t:.3f}, p (unilateral) = {p:.5f}')
print(f'Significativo (p < 0.05): {p < 0.05}')
print(f'd de Cohen = {d:.3f}')
try:
    wstat, wp = stats.wilcoxon(post, pre, alternative='greater')
    print(f'Wilcoxon (respaldo no paramétrico): W = {wstat:.3f}, p = {wp:.5f}')
except ValueError as e:
    print('Wilcoxon no aplicable:', e)

## 6. Gráficas

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot([pre, post], labels=['Pretest', 'Postest'])
ax[0].set_title('Distribución pretest vs postest')
ax[0].set_ylabel('Puntaje (0-20)')
x = np.arange(len(df))
ax[1].bar(x - 0.2, pre, 0.4, label='Pretest')
ax[1].bar(x + 0.2, post, 0.4, label='Postest')
ax[1].set_title('Puntaje por estudiante')
ax[1].set_xticks(x)
ax[1].set_xticklabels(df['codigo_estudiante'], rotation=90, fontsize=7)
ax[1].legend()
plt.tight_layout()
plt.show()

## 7. Interpretación (plantilla)

- Si **p < 0.05** en la t pareada → se rechaza H0: existe mejora significativa del rendimiento académico tras usar el tutor IA.
- Reportar en la tesis: media pretest, media postest, `t`, `gl = n−1`, `p`, **d de Cohen** e IC 95%.

> **RECORDATORIO:** con datos simulados este resultado **no es válido**. Sustituir por los datos reales del piloto (Sprint 8) y volver a ejecutar todo el notebook antes de la sustentación.